# Ectopic beats from QRS shape — trained on DS1, tested on DS2 (Notebook 11)
**Bektur Asilov · COMSE-23 · Ala-Too**

Notebook 10 showed RR intervals alone find < 10 % of ectopic beats. Here every beat found by our R-peak detector
(the firmware algorithm) is described by its timing **and its shape**, and a small model classifies it as
normal (N), supraventricular ectopic (S) or ventricular ectopic (V).

Rules fixed **before** the run:
* **inter-patient split** of de Chazal et al. (2004): train on DS1 (22 records), test on DS2 (22 other people);
  paced records excluded; nothing from DS2 is used for training or tuning;
* labels: AAMI classes, N = N, L, R, e, j; S = A, a, J, S; V = V, E; F and Q beats are left out of training and scoring;
  a detected beat takes the label of the annotated beat within 150 ms (unmatched beats are left out);
* features per beat (125 Hz, as on the device): RR before, RR after, mean of the 10 previous RR, their ratios;
  the beat ±250 ms (63 samples, every 2nd kept), scaled by the record's running median R amplitude;
  correlation with the median shape of the 20 previous beats; QRS width at half amplitude;
* model: scikit-learn `HistGradientBoostingClassifier(class_weight='balanced', max_iter=300, random_state=42)`;
* reported on DS2: sensitivity and PPV for V and S, and per record the ectopic burden (detected vs labelled),
  including "burden ≥ 10 %" — compared with the RR-only rule of Notebook 10 on the same DS2 records.

> Kaggle: Internet **On**, CPU.

In [ ]:
import os, time, json, subprocess, sys
import numpy as np, pandas as pd
from scipy.signal import butter, sosfilt, resample_poly
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
FS = 125
OUT_DIR = os.environ.get('OUT_DIR', '/kaggle/working')
DB = os.environ.get('MITDB_DIR')
RECS = [100,101,102,103,104,105,106,107,108,109,111,112,113,114,115,116,117,118,119,121,122,123,124,
        200,201,202,203,205,207,208,209,210,212,213,214,215,217,219,220,221,222,223,228,230,231,232,233,234]
if not DB:
    DB = '/kaggle/tmp/mitdb'; os.makedirs(DB, exist_ok=True)
    for r in RECS:
        for ext in ('hea', 'dat', 'atr'):
            for base in ('https://physionet-open.s3.amazonaws.com/mitdb/1.0.0/', 'https://physionet.org/files/mitdb/1.0.0/'):
                if subprocess.run(['curl', '-fsSL', '--max-time', '300', '-o', f'{DB}/{r}.{ext}', base + f'{r}.{ext}']).returncode == 0: break
            else:
                raise RuntimeError(f'download failed: {r}.{ext}')
print('records:', len(RECS))

In [ ]:
MA = 19; DECAY = 0.5 ** (1 / (2.0 * FS)); THR, REFR, WARM = 0.35, int(0.25 * FS), 2 * FS
try:
    from numba import njit
except ImportError:
    njit = lambda f: f

@njit
def detect_r(x):
    out = np.empty(len(x) // 20 + 10, np.int64); n = 0
    buf = np.zeros(MA); ssum = 0.0; prev = x[0]; pk = 0.0; last = -10**9; armed = True
    for i in range(len(x)):
        d = x[i] - prev; prev = x[i]; e = d * d
        ssum += e - buf[i % MA]; buf[i % MA] = e
        env = ssum / MA; pk = max(pk * DECAY, env)
        if i < WARM: continue
        if armed and env > THR * pk and i - last > REFR:
            out[n] = i; n += 1; last = i; armed = False
            if n >= len(out): break
        elif not armed and env < 0.5 * THR * pk:
            armed = True
    return out[:n]

def to_125(sig, fs):
    x = sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), np.nan_to_num(sig))
    from math import gcd
    g = gcd(int(fs), FS); return resample_poly(x, FS // g, int(fs) // g)

BEAT = set('NLRBAaJSVrFejnE/fQ?')

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]
CLS = {**{s: 0 for s in 'NLRej'}, **{s: 1 for s in 'AaJS'}, **{s: 2 for s in 'VE'}}
HALF = int(0.25 * FS)

def beats_features(rec):
    r = wfdb.rdrecord(f'{DB}/{rec}', channels=[0]); ann = wfdb.rdann(f'{DB}/{rec}', 'atr')
    x = to_125(r.p_signal[:, 0], r.fs); det = detect_r(x)
    ref_t = np.array([s * FS / r.fs for s, sym in zip(ann.sample, ann.symbol) if sym in BEAT])
    ref_y = np.array([CLS.get(sym, -1) for sym in ann.symbol if sym in BEAT])
    # align each detected beat to the largest |amplitude| within 60 ms, then match to labels
    pk = np.array([max(range(max(0, d - 8), min(len(x), d + 8)), key=lambda k: abs(x[k])) for d in det])
    j = np.clip(np.searchsorted(ref_t, pk), 1, len(ref_t) - 1)
    near = np.where(np.abs(ref_t[j - 1] - pk) < np.abs(ref_t[j] - pk), j - 1, j)
    y = np.where(np.abs(ref_t[near] - pk) <= 0.15 * FS, ref_y[near], -2)
    rr = np.diff(pk) / FS
    F, Y = [], []
    shapes = []
    for i in range(11, len(pk) - 1):
        a, b = pk[i] - HALF, pk[i] + HALF + 1
        if a < 0 or b > len(x): continue
        seg = x[a:b] - np.median(x[a:b])
        amp = np.median(np.abs([x[p] for p in pk[max(0, i - 20):i]])) or 1e-6
        seg = seg / amp
        prev = [s for s in shapes[-20:]]
        tmpl = np.median(prev, axis=0) if len(prev) >= 5 else seg
        corr = float(np.corrcoef(seg, tmpl)[0, 1]) if np.std(tmpl) > 0 and np.std(seg) > 0 else 0.0
        h = np.abs(seg[HALF - 12:HALF + 13]); width = float((h >= 0.5 * h.max()).sum()) / FS if h.max() > 0 else 0.0
        pre, post, loc = rr[i - 1], rr[i], rr[i - 11:i - 1].mean()
        F.append(np.concatenate([[pre, post, loc, pre / loc, post / pre, corr, width], seg[::2]]))
        Y.append(y[i]); shapes.append(seg)
    return np.array(F, np.float32), np.array(Y), len(det)

T0 = time.time(); DATA = {}
for rec in DS1 + DS2:
    if os.path.exists(f'{DB}/{rec}.dat'): DATA[rec] = beats_features(rec)
print({k: (len(v[0]), int((v[1] == 2).sum()), int((v[1] == 1).sum())) for k, v in list(DATA.items())[:5]}, f'{time.time()-T0:.0f} s')

In [ ]:
tr = [r for r in DS1 if r in DATA]; te = [r for r in DS2 if r in DATA]
Xtr = np.concatenate([DATA[r][0] for r in tr]); ytr = np.concatenate([DATA[r][1] for r in tr])
keep = ytr >= 0; Xtr, ytr = Xtr[keep], ytr[keep]
clf = HistGradientBoostingClassifier(class_weight='balanced', max_iter=300, random_state=42).fit(Xtr, ytr)
print('trained on', len(ytr), 'DS1 beats', np.bincount(ytr, minlength=3))

rows, allp, ally = [], [], []
for r in te:
    X, y, ndet = DATA[r]
    p = clf.predict(X) if len(X) else np.array([], int)
    m = y >= 0
    allp.append(p[m]); ally.append(y[m])
    rows.append(dict(record=r, beats=int(m.sum()), ref_S=int((y == 1).sum()), ref_V=int((y == 2).sum()),
                     det_S=int((p == 1).sum()), det_V=int((p == 2).sum()),
                     burden_ref=float(((y == 1) | (y == 2)).sum() / max(1, m.sum())),
                     burden_det=float(((p == 1) | (p == 2)).sum() / max(1, len(p)))))
P, Y = np.concatenate(allp), np.concatenate(ally)
def sp(c):
    tp = int(((P == c) & (Y == c)).sum()); return tp / max(1, (Y == c).sum()), tp / max(1, (P == c).sum())
R = pd.DataFrame(rows); R.to_csv(f'{OUT_DIR}/ectopy_qrs_ds2.csv', index=False)
from scipy.stats import spearmanr
hi_r, hi_d = R.burden_ref >= 0.10, R.burden_det >= 0.10
S = {'test_records': len(te), 'test_beats': int(len(Y)), 'V_sens_ppv': sp(2), 'S_sens_ppv': sp(1),
     'N_specificity': float(((P == 0) & (Y == 0)).sum() / max(1, (Y == 0).sum())),
     'confusion_rows_true_N_S_V_cols_pred': pd.crosstab(Y, P).values.tolist(),
     'burden_spearman': spearmanr(R.burden_ref, R.burden_det).correlation,
     'burden_mae_pct_points': float((R.burden_ref - R.burden_det).abs().mean() * 100),
     'records_burden_ge10': int(hi_r.sum()), 'found_ge10': int((hi_r & hi_d).sum()), 'false_ge10': int((~hi_r & hi_d).sum())}
json.dump(S, open(f'{OUT_DIR}/ectopy_qrs_summary.json', 'w'), indent=1, default=float)
print(json.dumps(S, indent=1, default=lambda v: round(float(v), 4)))
pd.set_option('display.width', 200)
print(R.assign(burden_ref=(R.burden_ref * 100).round(1), burden_det=(R.burden_det * 100).round(1)).to_string(index=False))